# poblar — ejecución en Colab

Este cuaderno instala la librería y la invoca. Los criterios están en el archivo de configuración; los productos, los controles y el manifiesto quedan en la carpeta de salida.

1. Completar los parámetros de la celda 3.
2. Ejecutar las celdas en orden.
3. Revisar `controles.csv` y `balance.csv` en la carpeta de la ejecución.

Entorno: el estándar de Colab (CPU) es suficiente para las etapas 0 y 1.

In [ ]:
# 1. Instalación
# Se clona el repositorio y se instala desde el clon: el código y las configuraciones
# son de la misma versión, y la ejecución registra el commit en entorno.txt.
# Para reproducir una versión exacta, indicar una etiqueta o un commit en VERSION.
VERSION = 'main'
!rm -rf /content/poblar_repo
!git clone -q https://github.com/AugustoBellanich/poblar /content/poblar_repo
!git -C /content/poblar_repo checkout -q {VERSION}
!pip install -q -e /content/poblar_repo
!git -C /content/poblar_repo log -1 --format='Versión: %h — %s (%cs)'

In [ ]:
# 2. Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Parámetros de la ejecución
from pathlib import Path

ANIO    = 2010                                                   # 2010 o 2022
CONFIG  = Path(f'/content/poblar_repo/aplicaciones/prais2026/configs/{ANIO}.yaml')
INSUMOS = Path('/content/drive/MyDrive/PRAIS/poblar/insumos')   # tabla_censoAAAA.xlsx, radios_AAAA.zip, regiones_PRAIS.gpkg
SALIDA  = Path('/content/drive/MyDrive/PRAIS/poblar/ejecuciones')
ETAPAS  = ['insumos', 'radios']

# Verificación previa: que estén los insumos que pide la configuración
import yaml
cfg = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
faltan = [i['ruta'] for i in cfg['insumos'].values() if not (INSUMOS / i['ruta']).exists()]
print('Faltan insumos:', faltan) if faltan else print('Insumos completos:', [i['ruta'] for i in cfg['insumos'].values()])

In [ ]:
# 4. Ejecución
# Si un control bloqueante falla, la ejecución se detiene y se informa el motivo;
# los controles y el balance quedan igual en la carpeta para revisarlos.
from poblar import ejecutar
from poblar.trazabilidad import ControlFallido

try:
    ejecucion = ejecutar(CONFIG, insumos=INSUMOS, salida=SALIDA, etapas=ETAPAS)
    CARPETA = ejecucion.carpeta
    print('\nEjecución completa:', CARPETA)
except ControlFallido as e:
    CARPETA = e.carpeta
    print('\nLa ejecución se detuvo:', e)
    print('Carpeta:', CARPETA)

In [ ]:
# 5. Controles y balance
import pandas as pd
pd.set_option('display.max_colwidth', 120)
controles = pd.read_csv(CARPETA / 'controles.csv')
print(f"Controles: {len(controles)} | sin cumplir: {(~controles['ok']).sum()}")
display(controles[['etapa', 'nombre', 'ok', 'detalle', 'bloqueante']])
display(pd.read_csv(CARPETA / 'balance.csv'))

## Comparación con la ejecución de referencia (opcional)

Compara `radios.parquet` con la caché del C2 original (`02_radios_curados_<año>.parquet`, en `PRAIS/Cuadernos/C2/<año>/cache`). La caché del 25/10/2026 se generó con la unión `left`, por lo que tiene un radio más (sin dato censal, población 0).

In [ ]:
# 6. Comparación radio por radio
from poblar.verificar import comparar_radios

REF = Path(f'/content/drive/MyDrive/PRAIS/Cuadernos/C2/{ANIO}/cache/02_radios_curados_{ANIO}.parquet')
if REF.exists():
    display(comparar_radios(CARPETA / '1_radios' / 'radios.parquet', REF))
else:
    print('No se encontró la referencia:', REF)